In [1]:
import pandas as pd
import numpy as np
import yfinance as yf

print("Notebook working")

Notebook working


**Download SPY data**

In [3]:
import pandas as pd

df = pd.read_csv("../../../data/raw/SPY.csv", parse_dates=["Date"])
df = df.set_index("Date")

df.head()

C:\Users\rawls.varghese\AppData\Local\Temp\ipykernel_19208\3314401044.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df = pd.read_csv("../../../data/raw/SPY.csv", parse_dates=["Date"])


,Open,High,Low,Close,Volume
Date,,,,,
2005-02-25,92.7949,93.8716,92.7097,93.6948,79289004
2005-02-28,93.4735,93.5952,92.6083,93.0544,89985785
2005-03-01,93.1845,93.7683,93.1845,93.5386,61658442
2005-03-02,93.1749,94.0766,93.0735,93.4927,83253671
2005-03-03,93.8716,94.0575,93.1203,93.5301,80491538


In [4]:
df.columns = [c.lower() for c in df.columns]

df.head()

,open,high,low,close,volume
Date,,,,,
2005-02-25,92.7949,93.8716,92.7097,93.6948,79289004
2005-02-28,93.4735,93.5952,92.6083,93.0544,89985785
2005-03-01,93.1845,93.7683,93.1845,93.5386,61658442
2005-03-02,93.1749,94.0766,93.0735,93.4927,83253671
2005-03-03,93.8716,94.0575,93.1203,93.5301,80491538


**Prediction Target**

In [5]:
df["ret_1d"] = df["close"].pct_change()
df["fwd_ret_5"] = df["close"].shift(-5) / df["close"] - 1
df["target"] = (df["fwd_ret_5"] > 0).astype(int)

df[["close","fwd_ret_5","target"]].head(10)

,close,fwd_ret_5,target
Date,,,
2005-02-25,93.6948,0.010806,1
2005-02-28,93.0544,0.018174,1
2005-03-01,93.5386,0.008851,1
2005-03-02,93.4927,-0.001813,0
2005-03-03,93.5301,0.000091,1
2005-03-04,94.7073,-0.019222,0
2005-03-07,94.7456,-0.013527,0
2005-03-08,94.3665,-0.017859,0
2005-03-09,93.3232,-0.015145,0


In [7]:
print("Rows:", len(df))
print("Start:", df.index.min())
print("End:", df.index.max())
print("Target prevalence:", df["target"].mean())

Rows: 5289
Start: 2005-02-25 00:00:00
End: 2026-03-06 00:00:00
Target prevalence: 0.5963320098317262


In [8]:
df_model = df.dropna().copy()

print("Rows after cleaning:", len(df_model))
df_model.head()

Rows after cleaning: 5283


,open,high,low,close,volume,ret_1d,fwd_ret_5,target
Date,,,,,,,,
2005-02-28,93.4735,93.5952,92.6083,93.0544,89985785,-0.006835,0.018174,1
2005-03-01,93.1845,93.7683,93.1845,93.5386,61658442,0.005203,0.008851,1
2005-03-02,93.1749,94.0766,93.0735,93.4927,83253671,-0.000491,-0.001813,0
2005-03-03,93.8716,94.0575,93.1203,93.5301,80491538,0.000400,0.000091,1
2005-03-04,94.1512,94.7819,93.9675,94.7073,72809892,0.012586,-0.019222,0


**Momentum Features**

In [ ]:
df["ret_5"] = df["close"].pct_change(5)
df["ret_10"] = df["close"].pct_change(10)
df["ret_20"] = df["close"].pct_change(20)

# This captures short term trend strength.
df[["close", "ret_5", "ret_10", "ret_20"]].head(15)

,close,ret_5,ret_10,ret_20
Date,,,,
2005-02-25,93.6948,NaN,NaN,NaN
2005-02-28,93.0544,NaN,NaN,NaN
2005-03-01,93.5386,NaN,NaN,NaN
2005-03-02,93.4927,NaN,NaN,NaN
2005-03-03,93.5301,NaN,NaN,NaN
2005-03-04,94.7073,0.010806,NaN,NaN
2005-03-07,94.7456,0.018174,NaN,NaN
2005-03-08,94.3665,0.008851,NaN,NaN
2005-03-09,93.3232,-0.001813,NaN,NaN


**Mean Reversion Feature**

In [ ]:
df["ma20"] = df["close"].rolling(20).mean()
df["ma_dist_20"] = (df["close"] - df["ma20"]) / df["ma20"]  # How far price is from 20 day average. This feature captures mean reversion pressure

df[["close", "ma20", "ma_dist_20"]].tail()

,close,ma20,ma_dist_20
Date,,,
2026-03-02,686.38,687.4015,-0.001486
2026-03-03,680.33,686.6475,-0.009200
2026-03-04,685.13,686.4275,-0.001890
2026-03-05,681.31,686.1835,-0.007102
2026-03-06,672.38,685.9215,-0.019742


**Volatility Feature**

In [ ]:
df["vol_ma20"] = df["volume"].rolling(20).mean()
df["volume_ratio"] = df["volume"] / df["vol_ma20"]

# High volume can signal institutional activity
df[["volume", "volume_ratio"]].tail()

,volume,volume_ratio
Date,,
2026-03-02,87477198,1.028164
2026-03-03,105003113,1.215780
2026-03-04,79182244,0.932316
2026-03-05,106606464,1.254182
2026-03-06,100686951,1.193615
